////////////////////////////////////////MY CODE///////////////////////////////////////////////

In [1]:
pip install sounddevice

Note: you may need to restart the kernel to use updated packages.


In [1]:
from __future__ import annotations

# Standard library imports
import json
import os
import wave
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Third‑party libraries
import numpy as np
import soundfile as sf
from jiwer import wer as jiwer_wer
from vosk import KaldiRecognizer, Model


In [2]:
# Optional microphone recording
import sounddevice as sd  # type: ignore
RECORDING_AVAILABLE = True


In [3]:
# Data structure for evaluation output
@dataclass
class EvalRow:
    Language: str
    File: str
    Reference: str
    Hypothesis: str
    WER: str


In [4]:
# Main ASR system class
class AirportASR:
    """
    Airport Virtual Assistant ASR System
    - Offline multi-language speech recognition using Vosk
    - Optional microphone recording (sounddevice)
    - Audio preprocessing for noisy environments
    - WER evaluation (jiwer with manual fallback)
    """

    def __init__(self, models_path: str | os.PathLike = "models") -> None:
        # Root directory containing Vosk models
        self.models_root = Path(models_path)
        self.models: Dict[str, Model] = {}

        # Mapping language code to model folder
        self._model_folders: Dict[str, str] = {
            "en": "vosk-model-en-us-0.22",
            "es": "vosk-model-small-es-0.42",
            "it": "vosk-model-small-it-0.22",
        }

        print("Initializing Airport ASR System...")
        self._load_all_models()

    
    # Model loading
    def _load_all_models(self) -> None:
        for lang, folder in self._model_folders.items():
            model_path = self.models_root / folder
            self._load_one_model(lang, model_path)

    def _load_one_model(self, lang: str, model_path: Path) -> None:
        try:
            print(f"Loading {lang.upper()} model from {model_path}...")
            self.models[lang] = Model(str(model_path))
            print(f"✓ {lang.upper()} model loaded successfully")
        except Exception as exc:
            print(f"✗ Error loading {lang.upper()} model: {exc}")

    def _load_all_models(self) -> None:
        # Load all configured language models
        for lang, folder in self._model_folders.items():
            model_path = self.models_root / folder
            self._load_one_model(lang, model_path)

    def _load_one_model(self, lang: str, model_path: Path) -> None:
        # Load a single Vosk model
        try:
            print(f"Loading {lang.upper()} model from {model_path}...")
            self.models[lang] = Model(str(model_path))
            print(f"✓ {lang.upper()} model loaded successfully")
        except Exception as exc:
            print(f"✗ Error loading {lang.upper()} model: {exc}")

   
    # Audio recording
    def record_audio(self, filename: str, duration: int = 5, sample_rate: int = 16000) -> Optional[str]:
        """
        Record audio from microphone and save as WAV.
        """
        if not RECORDING_AVAILABLE:
            print("Recording not available.")
            return None

        out_path = Path(filename)
        print(f"\n Recording for {duration} seconds...")

        # Capture audio
        frames = int(duration * sample_rate)
        recording = sd.rec(frames, samplerate=sample_rate, channels=1, dtype="float32")  # type: ignore
        sd.wait()  # type: ignore

        # Save as 16‑bit PCM WAV
        sf.write(str(out_path), recording, sample_rate, subtype="PCM_16")
        print(f"✓ Saved to: {out_path}")

        return str(out_path)

    # -----------------------------
    # Preprocessing
    # -----------------------------
    @staticmethod
    def _ensure_mono(audio: np.ndarray) -> np.ndarray:
        if audio.ndim > 1:
            return np.mean(audio, axis=1)
        return audio

    @staticmethod
    def _moving_average_lowpass(x: np.ndarray, sr: int, cutoff: int) -> np.ndarray:
        """
        Simple low-pass via moving average (keeps original behavior idea).
        """
        window = int(sr / cutoff)
        window = max(window, 3)
        if window % 2 == 0:
            window += 1

        kernel = np.ones(window, dtype=np.float32) / float(window)
        return np.convolve(x, kernel, mode="same")

    def preprocess_audio_enhanced(
        self,
        audio_path: str,
        output_path: Optional[str] = None,
        aggressive: bool = False,
    ) -> str:
        """
        Enhanced preprocessing with adjustable aggressiveness.

        Steps (kept functionally equivalent):
          - mono conversion (if needed)
          - DC offset removal
          - normalization
          - RMS-based gain adjustment
          - moving-average lowpass filter
          - noise gate (simple or dynamic)
          - final normalization + clipping
          - save as PCM_16 WAV

        Returns:
            path to the saved preprocessed file
        """
        in_path = Path(audio_path)
        print(f"Preprocessing: {in_path.name}")

        audio_data, sr = sf.read(str(in_path))
        audio_mono = self._ensure_mono(audio_data).astype(np.float32)

        # 1)reemove DC offset
        audio_mono = audio_mono - float(np.mean(audio_mono))

        # 2) normalize (peak)
        peak = float(np.max(np.abs(audio_mono))) if audio_mono.size else 0.0
        if peak > 0:
            audio_mono = (audio_mono / peak) * 0.95

        # 3) RMS-based gain
        rms = float(np.sqrt(np.mean(audio_mono**2))) if audio_mono.size else 0.0
        target_rms = 0.15
        if rms > 0:
            gain = np.clip(target_rms / rms, 0.5, 3.0)
            audio_mono = audio_mono * float(gain)

        # 4) Low-pass
        cutoff = 3200 if aggressive else 3800
        filtered = self._moving_average_lowpass(audio_mono, sr, cutoff=cutoff)

        # 5) noise gate
        if aggressive:
            abs_sorted = np.sort(np.abs(filtered))
            idx = int(len(abs_sorted) * 0.1) if len(abs_sorted) else 0
            dyn_thr = float(abs_sorted[idx]) if len(abs_sorted) else 0.0
            dyn_thr = max(dyn_thr, 0.01)

            quiet_mask = np.abs(filtered) < dyn_thr
            filtered[quiet_mask] *= 0.3
        else:
            thr = 0.015
            filtered[np.abs(filtered) < thr] *= 0.5

        # 6) Final normalize + clip
        peak2 = float(np.max(np.abs(filtered))) if filtered.size else 0.0
        if peak2 > 0:
            filtered = (filtered / peak2) * 0.9

        filtered = np.clip(filtered, -1.0, 1.0)

        # Decide output file namee
        if output_path is None:
            output_path = str(in_path.with_name(f"{in_path.stem}_preprocessed{in_path.suffix}"))

        sf.write(output_path, filtered, sr, subtype="PCM_16")
        print(f"  → Saved: {output_path}")

        return output_path

    # -----------------------------
    # Transcription
    # -----------------------------
    def _open_wave_mono(self, wav_path: str) -> Tuple[wave.Wave_read, str]:
        """
        Open a WAV file for reading. If it isn't mono, create a mono temp file and open that.
        Returns:
            (wave_handle, effective_path)
        """
        wf = wave.open(wav_path, "rb")
        if wf.getnchannels() == 1:
            return wf, wav_path

        # Convert to mono file (keeps behavior)
        wf.close()
        data, sr = sf.read(wav_path)
        mono = self._ensure_mono(data)
        mono_path = str(Path(wav_path).with_name(f"{Path(wav_path).stem}_mono.wav"))
        sf.write(mono_path, mono, sr, subtype="PCM_16")
        return wave.open(mono_path, "rb"), mono_path

    def transcribe(self, audio_path: str, language: str = "en", preprocess: bool = True, aggressive: bool = False) -> str:
        if language not in self.models:
            raise ValueError(f"Language '{language}' not supported or model not loaded")

        target_path = audio_path
        if preprocess:
            target_path = self.preprocess_audio_enhanced(audio_path, aggressive=aggressive)

        wf, effective_path = self._open_wave_mono(target_path)

        try:
            recognizer = KaldiRecognizer(self.models[language], wf.getframerate())
            recognizer.SetWords(True)

            chunks: List[str] = []
            while True:
                buf = wf.readframes(4000)
                if not buf:
                    break

                if recognizer.AcceptWaveform(buf):
                    part = json.loads(recognizer.Result())
                    txt = part.get("text", "")
                    if txt:
                        chunks.append(txt)

            final = json.loads(recognizer.FinalResult())
            final_txt = final.get("text", "")
            if final_txt:
                chunks.append(final_txt)

            return " ".join(chunks).strip()
        finally:
            wf.close()

# WER

    def calculate_wer(self, reference: str, hypothesis: str) -> float:
        """
        Calculate Word Error Rate (WER) as a percentage.
        Uses jiwer library for reliable WER computation.
        """
        # Handle empty reference cases
        if not reference.strip():
            return 100.0 if hypothesis.strip() else 0.0
    
        # Compute WER (case-insensitive)
        score = jiwer_wer(reference.lower(), hypothesis.lower())
        return round(float(score) * 100.0, 2)


# Evaluation
    def evaluate_test_set(
        self,
        audio_dir: str,
        transcriptions: Dict[str, str],
        language: str = "en",
        aggressive: bool = False,
    ) -> List[dict]:
        """
        Evaluate ASR on a directory of wav files, using a dict of filename -> reference text.
        Returns:
            list of dict rows (same shape as original behavior)
        """
        results: List[EvalRow] = []

        # Print evaluation header
        print("\n" + "=" * 70)
        print(f"Evaluating {language.upper()} Test Set")
        print("=" * 70)

        base = Path(audio_dir)

        # Loop through all reference transcriptions
        for fname, reference in transcriptions.items():
            wav_path = base / fname

            # Skip missing files
            if not wav_path.exists():
                print(f"⚠ File not found: {fname}")
                continue

            print(f"\nProcessing: {fname}")
            print(f"Reference: {reference}")

            # Transcribe audio file
            hypothesis = self.transcribe(str(wav_path), language=language, preprocess=True, aggressive=aggressive)
            print(f"Hypothesis: {hypothesis}")

            # Calculate WER
            wer_score = self.calculate_wer(reference, hypothesis)
            print(f"WER: {wer_score}%")

            # Store results
            results.append(
                EvalRow(
                    Language=language.upper(),
                    File=fname,
                    Reference=reference,
                    Hypothesis=hypothesis,
                    WER=f"{wer_score}%",
                )
            )

        # Keep the same outward structure: list of dicts
        return [asdict(r) for r in results]


In [5]:
# Custom sentence recording

def record_custom_sentences(asr: AirportASR) -> List[dict]:
    """
    Interactive recording session (keeps the same user-flow and output fields).
    """
    print("\n" + "=" * 70)
    print("CUSTOM RECORDING SESSION")
    print("=" * 70)

    # Check if microphone recording is available
    if not RECORDING_AVAILABLE:
        print("❌ Recording not available. Please install sounddevice:")
        print("   pip install sounddevice")
        return []

    # Sentences to record (reference, language)
    prompts: List[Tuple[str, str]] = [
        ("I need help finding my gate", "en"),
        ("Where is the nearest bathroom", "en"),
    ]

    collected: List[EvalRow] = []

    # Loop through prompts
    for idx, (reference, lang) in enumerate(prompts, start=1):
        print(f"\n--- Recording {idx}/{len(prompts)} ---")
        print(f"Please say: '{reference}'")
        input("Press ENTER when ready to record...")

        # Record audio
        out_name = f"your_sentence{idx}.wav"
        saved = asr.record_audio(out_name, duration=5, sample_rate=16000)
        if not saved:
            continue

        print("\nTranscribing...")
        hyp = asr.transcribe(saved, language=lang, preprocess=True, aggressive=True)
        wer_score = asr.calculate_wer(reference, hyp)

        # Display results
        print(f"\n Reference:  {reference}")
        print(f" Transcribed: {hyp}")
        print(f" WER: {wer_score}%")

        # Store result
        collected.append(
            EvalRow(
                Language=lang.upper(),
                File=out_name,
                Reference=reference,
                Hypothesis=hyp,
                WER=f"{wer_score}%",
            )
        )

    return [asdict(r) for r in collected]


In [6]:
#!/usr/bin/env python
# coding: utf-8

# In[3]:


# pip install scipy


# In[5]:
# Main program entry point
def main() -> List[dict]:
    # Initialize ASR system
    asr = AirportASR(models_path="models")

    # Ground truth
    english_transcriptions = {
        "checkin.wav": "where is the check-in desk",
        "parents.wav": "i have lost my parents",
        "suitcase.wav": "please i have lost my suitcase",
        "what_time.wav": "what time is my plane",
        "where.wav": "where are the restaurants and shops",
    }

    italian_transcriptions = {
        "checkin_it.wav": "dove e' il bancone",
        "parents_it.wav": "ho perso i miei genitori",
        "suitcase_it.wav": "per favore ho perso la mia valigia",
        "what_time_it.wav": "a che ora e' il mio aereo",
        "where_it.wav": "dove sono i ristoranti e i negozi",
    }

    spanish_transcriptions = {
        "checkin_es.wav": "dónde están los mostradores",
        "parents_es.wav": "he perdido a mis padres",
        "suitcase_es.wav": "por favor he perdido mi maleta",
        "what_time_es.wav": "a qué hora es mi avión",
        "where_es.wav": "dónde están los restaurantes y las tiendas",
    }

    all_results: List[dict] = []

    # Evaluate sets (same logic / thresholds)
    if "en" in asr.models:
        all_results.extend(
            asr.evaluate_test_set("Ex4_audio_files/EN", english_transcriptions, language="en", aggressive=True)
        )

    if "es" in asr.models:
        all_results.extend(
            asr.evaluate_test_set("Ex4_audio_files/ES", spanish_transcriptions, language="es", aggressive=False)
        )

    if "it" in asr.models:
        all_results.extend(
            asr.evaluate_test_set("Ex4_audio_files/IT", italian_transcriptions, language="it", aggressive=False)
        )

    # Optional custom recordings
    print("\n" + "=" * 70)
    print("Would you like to record custom sentences? (y/n)")
    answer = input().strip().lower()

    if answer == "y":
        all_results.extend(record_custom_sentences(asr))
    else:
        print("Skipping custom recordings.")

    # Summary table
    print("\n" + "=" * 70)
    print("FINAL RESULTS TABLE")
    print("=" * 70)
    print(f"{'Language':<12} {'File':<30} {'WER':<10}")
    print("-" * 70)

    for row in all_results:
        print(f"{row['Language']:<12} {row['File']:<30} {row['WER']:<10}")

    # Average WER by language
    print("\n" + "=" * 70)
    print("AVERAGE WER BY LANGUAGE")
    print("=" * 70)

    for lang in ("EN", "ES", "IT"):
        subset = [r for r in all_results if r["Language"] == lang]
        if not subset:
            continue

        values = [float(r["WER"].rstrip("%")) for r in subset]
        avg = float(np.mean(values))

        threshold = 25 if lang == "EN" else 35
        status = "✓ PASS" if avg < threshold else "✗ FAIL"
        print(f"{lang}: {avg:.2f}% (threshold: <{threshold}%) {status}")

    return all_results


if __name__ == "__main__":
    _ = main()
    print("\n✓ ASR System Evaluation Complete!")


# In[ ]:






Initializing Airport ASR System...
Loading EN model from models/vosk-model-en-us-0.22...


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=6
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:11:12:13:14:15
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from models/vosk-model-en-us-0.22/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from models/vosk-model-en-us-0.22/graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:294) Loading words from models/vosk-model-en-us-0.22/graph/words.txt
LOG (VoskAPI:ReadDataFiles():model.cc:310) Loading subtract G.fst model from models/vosk-model-en-us-0.22/rescore/G.fst
LOG (VoskAPI:ReadDataFiles():

✓ EN model loaded successfully
Loading ES model from models/vosk-model-small-es-0.42...


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=11 max-active=4000 lattice-beam=4
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from models/vosk-model-small-es-0.42/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from models/vosk-model-small-es-0.42/graph/HCLr.fst models/vosk-model-small-es-0.42/graph/Gr.fst


✓ ES model loaded successfully
Loading IT model from models/vosk-model-small-it-0.22...


...


CUSTOM RECORDING SESSION

--- Recording 1/2 ---
Please say: 'I need help finding my gate'


Press ENTER when ready to record... 



 Recording for 5 seconds...
✓ Saved to: your_sentence1.wav

Transcribing...
Preprocessing: your_sentence1.wav
  → Saved: your_sentence1_preprocessed.wav

 Reference:  I need help finding my gate
 Transcribed: after finding my gate
 WER: 50.0%

--- Recording 2/2 ---
Please say: 'Where is the nearest bathroom'


Press ENTER when ready to record... 



 Recording for 5 seconds...
✓ Saved to: your_sentence2.wav

Transcribing...
Preprocessing: your_sentence2.wav
  → Saved: your_sentence2_preprocessed.wav

 Reference:  Where is the nearest bathroom
 Transcribed: where is the nearest bathroom
 WER: 0.0%

FINAL RESULTS TABLE
Language     File                           WER       
----------------------------------------------------------------------
EN           checkin.wav                    40.0%     
EN           parents.wav                    20.0%     
EN           suitcase.wav                   0.0%      
EN           what_time.wav                  20.0%     
EN           where.wav                      0.0%      
ES           checkin_es.wav                 0.0%      
ES           parents_es.wav                 0.0%      
ES           suitcase_es.wav                0.0%      
ES           what_time_es.wav               83.33%    
ES           where_es.wav                   14.29%    
IT           checkin_it.wav                 50.0% 

////////////////////////////////////////MY CODE///////////////////////////////////////////////